# 03 — Eigenvalues, eigenvectors and SVD

**Needed for the assignment:** $\bigstar$ YES. **Study:** 1.5 h, **Practice:** 1 h.

In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

## Idea

An eigenvector of a square matrix $M$ is a direction that $M$ only stretches: $M\mathbf{v}=\lambda\mathbf{v}$. For a **symmetric** matrix (like $A^TA$ or a scatter matrix) the eigenvalues are real, the eigenvectors are perpendicular, and for $A^TA$ the eigenvalues are $\geq0$. **The single most used fact in this course:** the unit vector $\mathbf{x}$ that makes $\|A\mathbf{x}\|$ smallest is the eigenvector of $A^TA$ with the **smallest** eigenvalue (proof in topic 05).

## Key formulas

### Eigenvalues and Eigenvectors

**Eigenvalue equation**

## Eigenvalues and Eigenvectors

**Eigenvalue equation**

$$
M\mathbf{v}=\lambda\mathbf{v}
$$

The characteristic equation:

$$
\det(M-\lambda I)=0
$$

**For a $2\times2$ matrix:**

$$
\lambda^2-\operatorname{tr}(M)\lambda+\det(M)=0
$$

## Singular Value Decomposition (SVD)

$$
A=U\Sigma V^T
$$

where:

$$
U^TU=I,\qquad V^TV=I
$$

$$
\Sigma=\operatorname{diag}(\sigma_1,\sigma_2,\ldots)
$$

Singular values are ordered:

$$
\sigma_1\geq\sigma_2\geq\cdots\geq0
$$

**Relationship between SVD and eigenvalues:**

- Columns of $V$ are eigenvectors of $A^TA$.
- Squared singular values are eigenvalues of $A^TA$:

$$
\sigma_i^2=\lambda_i(A^TA)
$$

## Important NumPy notation

The eigenvector corresponding to the smallest eigenvalue of $A^TA$ is:

- The last column of $V$.
- The last row of $V^T$.
- `Vt[-1]` in NumPy, when using `U, S, Vt = np.linalg.svd(A)`.

$$
\boxed{\text{Smallest eigenvector of }A^TA=V^T[-1]}
$$

# Worked example (on paper)

$M=\begin{bmatrix}2&1\\1&2\end{bmatrix}$.

**1. Eigenvalues.** $\text{tr}M=4$, $\det M=3$:   $\lambda^2-4\lambda+3=0\Rightarrow(\lambda-1)(\lambda-3)=0\Rightarrow\lambda_1=1,\ \lambda_2=3$.

**2. Eigenvector for $\lambda=1$.** $(M-I)\mathbf{v}=\begin{bmatrix}1&1\\1&1\end{bmatrix}\mathbf{v}=\mathbf{0}\Rightarrow v_1+v_2=0\Rightarrow \mathbf{v}=\tfrac1{\sqrt2}(1,-1)$.

**3. Eigenvector for $\lambda=3$.** $(M-3I)\mathbf{v}=\begin{bmatrix}-1&1\\1&-1\end{bmatrix}\mathbf{v}=\mathbf{0}\Rightarrow v_1=v_2\Rightarrow\mathbf{v}=\tfrac1{\sqrt2}(1,1)$.   Check: they are perpendicular, $(1,-1)\cdot(1,1)=0$ ✓.

**4. Link to SVD.** Take $A=\begin{bmatrix}1&1\\0&1\\1&0\end{bmatrix}$. Then $A^TA=\begin{bmatrix}1+0+1&1+0+0\\1+0+0&1+1+0\end{bmatrix}=\begin{bmatrix}2&1\\1&2\end{bmatrix}=M$. So the singular values of $A$ are $\sqrt3\approx1.732$ and $\sqrt1=1$, and the rows of $V^T$ are $\pm\tfrac1{\sqrt2}(1,1)$ and $\pm\tfrac1{\sqrt2}(1,-1)$.

**Note:** an eigenvector is only defined up to sign (and scale). $(1,-1)$ and $(-1,1)$ are the same answer.

# The same in Python / NumPy

In [2]:
import numpy as np

M = np.array([[2.0, 1.0],
              [1.0, 2.0]])

# eig: eigenvectors are the COLUMNS, order is not guaranteed
val, vec = np.linalg.eig(M)
print("dtype from eig:", val.dtype)   # may be complex even for symmetric M!
val, vec = val.real, vec.real            # imaginary parts are 0 -> drop them
print("eig values :", val)
print("eig vectors (columns):\n", np.round(vec, 4))

# smallest eigenvalue -> its eigenvector (the pattern used everywhere in the course)
i = np.argmin(val)
v = vec[:, i]
print("smallest:", val[i], " v =", np.round(v, 4))
print("check M v - lambda v =", np.round(M @ v - val[i] * v, 10))

# eigh: for symmetric matrices, values sorted ascending -> column 0 is the smallest
val_h, vec_h = np.linalg.eigh(M)
print("eigh values:", val_h)

# SVD of A whose A^T A is exactly M
A = np.array([[1.0, 1.0],
              [0.0, 1.0],
              [1.0, 0.0]])
print("A^T A =\n", A.T @ A)
U, S, Vt = np.linalg.svd(A)
print("singular values:", np.round(S, 4), " squared:", np.round(S**2, 4))
print("rows of Vt (= eigenvectors of A^T A):\n", np.round(Vt, 4))
print("last row of Vt (smallest):", np.round(Vt[-1], 4))

dtype from eig: complex128
eig values : [3. 1.]
eig vectors (columns):
 [[ 0.7071 -0.7071]
 [ 0.7071  0.7071]]
smallest: 1.0  v = [-0.7071  0.7071]
check M v - lambda v = [0. 0.]
eigh values: [1. 3.]
A^T A =
 [[2. 1.]
 [1. 2.]]
singular values: [1.7321 1.    ]  squared: [3. 1.]
rows of Vt (= eigenvectors of A^T A):
 [[-0.7071 -0.7071]
 [ 0.7071 -0.7071]]
last row of Vt (smallest): [ 0.7071 -0.7071]


## NumPy pitfalls

-   `np.linalg.eig` returns eigenvectors as **columns**: use `vec[:, i]`, not `vec[i]`.

-   `eig` does not sort, so use `np.argmin(val)`. `eigh` (symmetric only) sorts ascending, so column 0 is the smallest.

-   In recent NumPy versions `eig` can return a **complex** dtype even when everything is real. Take `.real`, or use `eigh`.

-   `svd` returns $V^T$ (`Vt`): the eigenvectors are its **rows**, and the smallest one is `Vt[-1]`.

## Where this is used in the assignment

Refining a plane, fitting a sphere, the local normals of the range image: all of them are "take the smallest eigenvector".

## Course / lab link

Lab 6 / Practice: `eig_val, eig_vec = np.linalg.eig(A.T@A)`, `h = eig_vec[:, np.argmin(eig_val)]`. Zhang's method uses `svd` and `vh[-1]`. Both give the same vector.

## Practice

Write your solution in the empty cell under each exercise. The **expected result** tells you what you should get, not how to get it. Grey *provided data* cells are already filled in; run them first.

### Exercise 1

Find the eigenvalues and eigenvectors of $\begin{bmatrix}5&2\\2&2\end{bmatrix}$ by hand. Are the eigenvectors perpendicular?

> **Expected result:** $\lambda=1$ with $\mathbf v\propto(1,-2)$, and $\lambda=6$ with $\mathbf v\propto(2,1)$; normalised $\pm(0.4472,-0.8944)$ and $\pm(0.8944,0.4472)$. Yes, their dot product is 0.

In [3]:
# Exercise 1: your solution


### Exercise 2

Check with `eig` and with `eigh`. Which column holds the smallest eigenvalue in each?

> **Expected result:** `eigh` returns `[1., 6.]`, so the smallest is column 0. `eig` has no guaranteed order, so use `np.argmin`.

In [4]:
# Exercise 2: your solution


### Exercise 3

Build any $4\times2$ matrix $A$. Show that `svd(A)`'s `S**2` equals the eigenvalues of `A.T@A` from `eigh` (in reverse order).

> **Expected result:** `np.allclose(S**2, eigh_vals[::-1])` is `True`.

In [5]:
# Exercise 3: your solution


### Exercise 4

What are the eigenvalues of the $2\times2$ rotation $\begin{bmatrix}0&-1\\1&0\end{bmatrix}$? Why is this not a contradiction?

> **Expected result:** $\pm i$ (complex). The matrix is not symmetric, so the real-eigenvalue rule does not apply.

In [6]:
# Exercise 4: your solution
